In [1]:
from py_clob_client.constants import POLYGON
from py_clob_client.client import ClobClient
from py_clob_client.clob_types import OrderArgs, PostOrdersArgs, OrderType, BalanceAllowanceParams, AssetType, OpenOrderParams, BookParams, MarketOrderArgs
from py_clob_client.order_builder.constants import BUY
from datetime import datetime, timezone, timedelta
import requests
import time
import math
import signal 

import os
from dotenv import load_dotenv

load_dotenv()
private_key = os.getenv("PRIVATE_KEY")
funder = os.getenv("funder")
tg_token = os.getenv("tg_token")

host = "https://clob.polymarket.com"
chain_id = POLYGON
chat_id = '-1002321264842'
client = ClobClient(host, key=private_key, chain_id=chain_id, signature_type=1, funder=funder)
api_creds = client.create_or_derive_api_creds()
client.set_api_creds(api_creds)
#Hyperparameters
fetch_count = 500



In [ ]:
def get_YES_NO_And_Condition(fetch_count: int = 20, date: str = "2026-01-01") -> list:
    """Fetch events data from Polymarket API"""
    import requests
    
    r = requests.get(f"https://gamma-api.polymarket.com/events?limit={fetch_count}&closed=false&start_date_min={date}T00:00:00Z")
    response = r.json()
    all_events = []

    for event in response:
        if event['enableNegRisk'] :
            title = event["title"]
            current_event = [title, [], [], [], [],[],[], []]

            for x in event['markets']:
                if x["active"] and not x['closed']:
                    YES_ID = x['clobTokenIds'].split(',')[0].replace('[','').replace(']','').replace('\'','').replace('"','').replace(' ','')
                    NO_ID = x['clobTokenIds'].split(',')[1].replace('[','').replace(']','').replace('\'','').replace('"','').replace(' ','')
                    condition_id = x["conditionId"]
                    rewards_daily_rate = x.get("clobRewards", [{}])[0].get("rewardsDailyRate", 0)
                    rewards_min_size = x["rewardsMinSize"]
                    rewards_max_spread = x["rewardsMaxSpread"]
                    min_tick_size = x["orderPriceMinTickSize"]
                    current_event[1].append(YES_ID)
                    current_event[2].append(NO_ID)
                    current_event[3].append(condition_id)
                    current_event[4].append(rewards_daily_rate)
                    current_event[5].append(rewards_min_size)
                    current_event[6].append(rewards_max_spread)
                    current_event[7].append(float(min_tick_size))
            all_events.append(current_event)

    return all_events

In [2]:
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Any, Set
import time
from enum import Enum
import json
import requests
import math
import threading
import websocket
from queue import Queue

# Assuming Polymarket SDK imports (adjust based on your actual SDK)
# from polymarket_sdk import OrderArgs, PostOrdersArgs, OrderType, BookParams, BalanceAllowanceParams, AssetType

class OrderSide(Enum):
    BUY = "BUY"
    SELL = "SELL"

class OrderStatus(Enum):
    PENDING = "pending"
    MATCHED = "matched"
    CANCELLED = "cancelled"
    EXPIRED = "expired"
    FAILED = "failed"


@dataclass
class MarketToken:
    """Represents a single outcome token (YES or NO)"""
    condition_id: str
    token_id: str  # YES_ID or NO_ID
    token_type: str  # "YES" or "NO"
    best_bid: Optional[float] = None
    best_ask: Optional[float] = None
    
    def to_dict(self) -> Dict:
        return {
            "condition_id": self.condition_id,
            "token_id": self.token_id,
            "token_type": self.token_type,
            "best_bid": self.best_bid,
            "best_ask": self.best_ask,
        }

@dataclass
class MarketRewards:
    """Market rewards and trading limits"""
    rates: float = 0.0  # Single number, 0 if None
    min_size: float = 0.0
    max_spread: float = 0.0
    
    def to_dict(self) -> Dict:
        return {
            "rates": self.rates,
            "min_size": self.min_size,
            "max_spread": self.max_spread
        }
    
@dataclass 
class Market:
    """Represents a single market within an event"""
    market_id: str  # Using condition_id as market_id
    title: str
    condition_id: str
    yes_token: MarketToken  # YES outcome
    no_token: MarketToken   # NO outcome
    min_tick: float
    rewards: MarketRewards = field(default_factory=MarketRewards)  # Add rewards data
    last_price: Optional[float] = None
    
    
    def get_token(self, token_type: str) -> MarketToken:
        """Get YES or NO token by type"""
        if token_type.upper() == "YES":
            return self.yes_token
        elif token_type.upper() == "NO":
            return self.no_token
        else:
            raise ValueError(f"Invalid token type: {token_type}")
    
    def get_other_token(self, token_id: str) -> Optional[MarketToken]:
        """Given a token ID, return the opposite token (YES ↔ NO)"""
        if token_id == self.yes_token.token_id:
            return self.no_token
        elif token_id == self.no_token.token_id:
            return self.yes_token
        return None
    
    def to_dict(self) -> Dict:
        return {
            "market_id": self.market_id,
            "title": self.title,
            "condition_id": self.condition_id,
            "yes_token": self.yes_token.to_dict(),
            "no_token": self.no_token.to_dict(),
            "rewards": self.rewards.to_dict(),
            "last_price": self.last_price
        }

@dataclass
class Event:
    """Represents a complete event with multiple markets"""
    event_id: str  # Using hash of title + first condition_id
    title: str
    markets: Dict[str, Market]  # market_id -> Market
    
    def get_all_tokens(self) -> List[MarketToken]:
        """Get all tokens across all markets in this event"""
        tokens = []
        for market in self.markets.values():
            tokens.extend([market.yes_token, market.no_token])
        return tokens
    
    def get_all_no_tokens(self) -> List[str]:
        """Get all NO tokens across all markets in this event"""
        tokens = []
        for market in self.markets.values():
            tokens.append(market.no_token.token_id)
        return tokens
    
    def get_market_by_token(self, token_id: str) -> Optional[Market]:
        """Find which market contains a specific token"""
        for market in self.markets.values():
            if (market.yes_token.token_id == token_id or 
                market.no_token.token_id == token_id):
                return market
        return None
    
    def to_dict(self) -> Dict:
        return {
            "event_id": self.event_id,
            "title": self.title,
            "markets": {k: v.to_dict() for k, v in self.markets.items()},
        }

@dataclass
class OrderRecord:
    """Track an active order"""
    order_id: str
    asset_id: str  # This is token_id from MarketToken
    market_id: str  # Which market this order belongs to
    event_id: str   # Which event this order belongs to
    price: float
    size: float
    initial_market_bid: float
    initial_market_ask: float
    side: OrderSide
    token_type: str  # "YES" or "NO"
    created_at: float
    expiration: float
    status: OrderStatus = OrderStatus.PENDING
    matched_amount: Optional[float] = None
    tx_hash: Optional[str] = None
    last_checked: Optional[float] = None
    raw_response: Optional[Dict] = None
    
    def is_expired(self) -> bool:
        return time.time() > self.expiration
    
    def to_dict(self) -> Dict:
        return {
            "order_id": self.order_id,
            "asset_id": self.asset_id,
            "market_id": self.market_id,
            "event_id": self.event_id,
            "price": self.price,
            "size": self.size,
            "initial_market_bid": self.initial_market_bid,
            "initial_market_ask": self.initial_market_ask,
            "side": self.side.value,
            "token_type": self.token_type,
            "created_at": self.created_at,
            "expiration": self.expiration,
            "status": self.status.value,
            "matched_amount": self.matched_amount,
            "tx_hash": self.tx_hash,
            "last_checked": self.last_checked
        }

In [ ]:
class TradingManager:
    """Main manager to handle events, markets, and orders"""
    
    def __init__(self, client=None, api_key="", api_secret="", api_passphrase=""):
        self.events: Dict[str, Event] = {}  # event_id -> Event
        self.markets: Dict[str, Market] = {}  # market_id -> Market
        self.tokens: Dict[str, MarketToken] = {}  # token_id -> MarketToken
        self.active_orders: Dict[str, OrderRecord] = {}  # asset_id(token_id) -> OrderRecord
        self.order_id_mapping: Dict[str, str] = {}  # order_id -> asset_id
        self.client = client  # Optional Polymarket client
        self.active_events: Set[str] = set()          # event_ids that have at least one active order
        self.recently_checked_events: Dict[str, float] = {} 
        self.last_full_scan = 0.0
        self.FULL_SCAN_INTERVAL_SECONDS = 90   # 1.5 minutes
        self.api_key = api_key
        self.api_secret = api_secret
        self.api_passphrase = api_passphrase
        self.wss_url = "wss://ws-subscriptions-clob.polymarket.com/ws/user"
        self.ws = None
        self.ws_thread = None
        self.ws_running = False
        self.message_queue = Queue()   # for decoupling WS thread from main logic
        self.last_ws_message_time = time.time()

        self.market_ws = None
        self.market_ws_thread = None
        self.market_ws_running = False
        self.market_wss_url = "wss://ws-subscriptions-clob.polymarket.com/ws/market"

        # token_id -> {"shares": float, "avg_cost": float, "total_cost": float}
        self.positions: Dict[str, Dict] = {}

        self.last_market_refresh = 0.0
        self.MARKET_REFRESH_INTERVAL_SECONDS = 3600 # refresh every hour

        # Start WS if credentials provided
        if self.api_key and self.api_secret and self.api_passphrase:
            self.start_websocket()
            self.start_market_websocket()

    def should_refresh_markets(self) -> bool:
        now = time.time()
        if now - self.last_market_refresh >= self.MARKET_REFRESH_INTERVAL_SECONDS:
            self.last_market_refresh = now
            return True
        return False

    def refresh_markets(self, fetch_count: int = 500):
        """Fetch latest market data and merge into existing state without disrupting active orders"""
        print("[refresh_markets] Fetching latest market data...")
        try:
            raw_data = get_YES_NO_And_Condition(fetch_count=fetch_count)
        except Exception as e:
            print(f"[refresh_markets] Fetch failed: {e}")
            return

        # Track what we had before
        existing_event_ids = set(self.events.keys())

        # Collect incoming event ids
        incoming_event_ids = set()
        for event_data in raw_data:
            title = event_data[0]
            condition_ids = event_data[3]
            if condition_ids:
                incoming_event_ids.add(title)

        # Find newly closed events — in our records but not in fresh fetch
        closed_event_ids = existing_event_ids - incoming_event_ids
        for event_id in closed_event_ids:
            event = self.events.get(event_id)
            if not event:
                continue

            # If we have active orders in a now-closed event, cancel them
            active_in_event = [
                token_id for token_id, order in self.active_orders.items()
                if order.event_id == event_id
            ]
            if active_in_event:
                print(f"[refresh_markets] Event '{event_id}' is now closed — cancelling {len(active_in_event)} orders")
                self.cancel_orders(active_in_event, self.client)

            # Remove from tracking
            for market_id, market in event.markets.items():
                self.tokens.pop(market.yes_token.token_id, None)
                self.tokens.pop(market.no_token.token_id, None)
                self.markets.pop(market_id, None)

            del self.events[event_id]
            self.active_events.discard(event_id)
            print(f"[refresh_markets] Removed closed event '{event_id}'")

        # Merge new and updated events
        self.process_api_response(raw_data)
        print(f"[refresh_markets] Done — {len(self.events)} events, "
            f"{len(self.markets)} markets, {len(self.tokens)} tokens")




    def start_websocket(self):
        """Start WebSocket in a daemon thread, adapted from sample"""

        def on_message(ws, message):
            self.last_ws_message_time = time.time()
            print(f"[WS] RAW: {message}")
            if not message.strip():
                return
            try:
                data = json.loads(message)
                self.process_ws_message(data)
            except json.JSONDecodeError as e:
                print(f"[WS] JSON decode failed: {e}")
            except Exception as e:
                print(f"[WS] Processing error: {e}")

        def on_error(ws, error):
            print("[WS] Error:", error)

        def on_close(ws, close_status_code, close_msg):
            print(f"[WS] Closed: {close_status_code} - {close_msg}")
            self.ws_running = False

        def on_open(ws):
            print("[WS] Connected → subscribing to user channel")
            auth = {
                "apiKey": self.api_key,
                "secret": self.api_secret,
                "passphrase": self.api_passphrase
            }
            sub_msg = {"markets": [], "type": "user", "auth": auth}
            ws.send(json.dumps(sub_msg))

            # ✅ Ping thread launched here — ws is guaranteed live
            def ping_thread():
                while self.ws_running:
                    try:
                        ws.send("PING")
                    except Exception as e:
                        print(f"[WS] Ping failed: {e}")
                        break
                    time.sleep(10)

            threading.Thread(target=ping_thread, daemon=True).start()
            print("[WS] Ping thread started")

        def run_ws():
            while self.ws_running:
                try:
                    self.ws = websocket.WebSocketApp(
                        self.wss_url,
                        on_open=on_open,
                        on_message=on_message,
                        on_error=on_error,
                        on_close=on_close
                    )
                    self.ws.run_forever(ping_interval=30, ping_timeout=10)
                except Exception as e:
                    print("[WS] Reconnecting in 5s...", e)
                    time.sleep(5)

        self.ws_running = True
        self.ws_thread = threading.Thread(target=run_ws, daemon=True)
        self.ws_thread.start()
        print("[WS] Thread started")

    def start_market_websocket(self):
        """Subscribe to market channel for price_change updates on relevant NO tokens"""

        def on_message(ws, message):
            self.last_ws_message_time = time.time()
            print(f"[Market WS] RAW: {message}")
            try:
                data = json.loads(message)
                self.process_market_message(data)
            except Exception as e:
                print("[Market WS] Error processing:", e)

        def on_error(ws, error):
            print("[Market WS] Error:", error)

        def on_close(ws, code, msg):
            print(f"[Market WS] Closed: {code} - {msg}")
            self.market_ws_running = False

        def on_open(ws):
            print("[Market WS] Connected → re-subscribing to active tokens")

            # Re-subscribe to all NO tokens for currently active events on reconnect
            active_token_ids = []
            for event_id in self.active_events:
                event = self.events.get(event_id)
                if event:
                    active_token_ids.extend(event.get_all_no_tokens())

            if active_token_ids:
                sub_msg = {"assets_ids": active_token_ids, "operation": "subscribe"}
                ws.send(json.dumps(sub_msg))
                print(f"[Market WS] Re-subscribed to {len(active_token_ids)} tokens")
            else:
                print("[Market WS] No active tokens to subscribe to on connect")

        def run_market_ws():
            while self.market_ws_running:
                try:
                    self.market_ws = websocket.WebSocketApp(
                        self.market_wss_url,
                        on_open=on_open,
                        on_message=on_message,
                        on_error=on_error,
                        on_close=on_close
                    )
                    self.market_ws.run_forever(ping_interval=30, ping_timeout=10)
                except Exception as e:
                    print("[Market WS] Reconnecting in 5s...", e)
                    time.sleep(5)

        self.market_ws_running = True
        self.market_ws_thread = threading.Thread(target=run_market_ws, daemon=True)
        self.market_ws_thread.start()
        print("[Market WS] Thread started")

    def shutdown(self):
        """Graceful shutdown — cancel all orders then close both websockets"""
        print("\n[shutdown] Starting graceful shutdown...")

        # 1. Cancel all active orders first before closing connections
        all_token_ids = list(self.active_orders.keys())
        if all_token_ids:
            print(f"[shutdown] Cancelling {len(all_token_ids)} active orders...")
            self.cancel_orders(all_token_ids, self.client)
        else:
            print("[shutdown] No active orders to cancel")

        # 2. Stop user websocket
        print("[shutdown] Closing user WebSocket...")
        self.ws_running = False
        if self.ws:
            self.ws.close()

        # 3. Stop market websocket
        print("[shutdown] Closing market WebSocket...")
        self.market_ws_running = False
        if self.market_ws:
            self.market_ws.close()

        print("[shutdown] Goodbye!")
        sys.exit(0)

    def register_signal_handler(self):
        """Register SIGINT handler — must be called from main thread"""
        def signal_handler(sig, frame):
            print("\n[signal_handler] Keyboard interrupt received (SIGINT)")
            self.shutdown()

        signal.signal(signal.SIGINT, signal_handler)
        print("[signal_handler] Registered — press Ctrl+C to stop gracefully")

    def process_ws_message(self, data: dict):
        """Update OrderRecord from WS messages"""
        event_type = data.get("event_type")

        if event_type == "order":
            order_id = data.get("id")
            token_id = self.order_id_mapping.get(order_id)
            if not token_id:
                return
            size_matched = float(data.get("size_matched", 0))
            status_type = data.get("type")  # PLACEMENT / UPDATE / CANCELLATION

            if token_id in self.active_orders:
                order = self.active_orders[token_id]
                order.matched_amount = size_matched
                order.last_checked = time.time()

                if status_type == "CANCELLATION":
                    order.status = OrderStatus.CANCELLED
                    print(f"[WS] Order {order_id} CANCELLED for token {token_id}")
                    # Cleanup
                    self.order_id_mapping.pop(order_id, None)
                    del self.active_orders[token_id]
                elif status_type == "UPDATE" and size_matched > 0:
                    print(f"[WS] Partial fill detected: {size_matched} / {order.size} for {token_id}")
                    # Trigger hedge if significant fill
                    event = self.events.get(order.event_id)
                    if event:
                        self.hedging(event, self.client)
                order.raw_response = data

        elif event_type == "trade":
            status = data.get("status")
            maker_orders = data.get("maker_orders", [])
            for mo in maker_orders:
                order_id = mo.get("order_id")
                token_id = self.order_id_mapping.get(order_id)
                if not token_id:
                    continue
                matched = float(mo.get("matched_amount", 0))

                if token_id in self.active_orders:
                    order = self.active_orders[token_id]
                    order.matched_amount = (order.matched_amount or 0) + matched  # incremental
                    order.last_checked = time.time()

                    if status in ("MATCHED", "MINED", "CONFIRMED"):
                        print(f"[WS] Trade {status} for order {order_id} - matched {matched}")
                        # Trigger hedge
                        event = self.events.get(order.event_id)
                        if event:
                            self.hedging(event, self.client)
                    elif status in ("FAILED", "RETRYING"):
                        print(f"[WS] Trade issue ({status}) for {token_id}")
                        # Alert Telegram if failed
                        message = f"Trade Failed for order {order_id} in event {event.title if event else 'unknown'}"
                        Telegram_url = f"https://api.telegram.org/bot{tg_token}/sendMessage?chat_id={chat_id}&text={message}"
                        requests.get(Telegram_url)

   

    def process_market_message(self, data: Any):
            """Handle incoming market channel messages safely"""
            if not isinstance(data, (dict, list)):
                if isinstance(data, str) and data.strip() in ("PING", "pong", ""):
                    return  # heartbeat
                print(f"[Market WS] Unexpected non-JSON: {type(data)} → {data}")
                return

            messages = data if isinstance(data, list) else [data]

            for msg in messages:
                if not isinstance(msg, dict):
                    continue

                event_type = msg.get("event_type")
                if not event_type:
                    print(f"[Market WS] Message missing 'event_type': {msg}")
                    continue

                if event_type == "price_change":
                    token_id = msg.get("asset_id")
                    if not token_id:
                        continue
                    # If there are no active orders for this token's event, ignore
                    event = self.get_event_by_token(token_id)
                    if not event:
                        continue
                    
                    has_active_in_event = any(
                    o.event_id == event.event_id for o in self.active_orders.values()
                    )
                    if not has_active_in_event:
                        continue

                    #Best Ask of Updated
                    new_best_ask = float(msg['asks'][-1]['price'])

                    # Determine an initial ask to compare against:
                    # Prefer the tracked order initial ask if this token has an active order,
                    # otherwise fall back to the stored token.best_ask snapshot (if present).
                    if token_id in self.active_orders:
                        order = self.active_orders[token_id]
                        initial_ask = order.initial_market_ask
                    else:
                        token_obj = self.tokens.get(token_id)
                        initial_ask = token_obj.best_ask if token_obj else None

                        # Worsened beyond threshold => cancel all orders in the event
                    if new_best_ask > initial_ask + 0.002:  # adjust threshold
                        print(f"[Market WS] ASK WORSENED on {token_id}: {initial_ask:.4f} → {new_best_ask:.4f}")
                        # Cancel all maker orders for this event
                        token_ids_for_event = event.get_all_no_tokens()
                        self.cancel_orders(token_ids_for_event, self.client)
                        print("[Market WS] Cancelled all orders for event due to worsened ask")

                    # update token snapshot
                    if token_id in self.tokens:
                        self.tokens[token_id].best_ask = new_best_ask

                elif event_type == "book":
                    # Optional: you can use book updates to update internal price cache if you want
                    # For now just log (or ignore silently)
                    token_id = msg.get("asset_id")
                    if token_id in self.active_orders:
                        print(f"[Market WS] Book update for watched token {token_id} — best ask size: {msg.get('asks', [{}])[0].get('size', 'N/A')}")
                    # You could add logic here to cancel if liquidity dried up severely

                elif event_type in ("subscribed", "unsubscribed"):
                    print(f"[Market WS] Subscription ack: {msg}")

                else:
                    print(f"[Market WS] Unknown event_type '{event_type}': {msg}")


    def subscribe_to_tokens_ids(self, event: Event):
        ws = self.market_ws
        assets_ids = event.get_all_no_tokens()
        sub_msg = {"assets_ids": list(assets_ids), "operation": "subscribe"}
        try:
            ws.send(json.dumps(sub_msg))
            print(f"[Market WS] Updated subscription to {len(assets_ids)} NO tokens")
        except Exception as e:
            print("[Market WS] Subscribe failed:", e)

    def unsubscribe_to_tokens_ids(self, event: Event):
        ws = self.market_ws
        assets_ids = event.get_all_no_tokens()
        sub_msg = {"assets_ids": list(assets_ids), "operation": "unsubscribe"}
        try:
            ws.send(json.dumps(sub_msg))
            print(f"[Market WS] Updated subscription to {len(assets_ids)} NO tokens")
        except Exception as e:
            print("[Market WS] Unsubscribe failed:", e)

    def check_and_deactivate_event(self, event_id: str):
        """Remove event from active_events and unsubscribe from market WS if no remaining active orders"""
        has_active_orders = any(
            o.event_id == event_id 
            for o in self.active_orders.values()
        )
        if not has_active_orders:
            self.active_events.discard(event_id)
            self.recently_checked_events.pop(event_id, None)  # ✅ clean up timestamp too
            print(f"[active_events] Event '{event_id}' deactivated — no remaining orders")

            # ✅ Unsubscribe from market WS to stop receiving irrelevant price updates
            event = self.events.get(event_id)
            if event and self.market_ws and self.market_ws_running:
                self.unsubscribe_to_tokens_ids(event)
                print(f"[active_events] Unsubscribed market WS tokens for event '{event_id}'")
            else:
                print(f"[active_events] Market WS not running — no unsubscribe needed")


    def stop_websocket(self):
        self.ws_running = False
        if self.ws:
            self.ws.close()

    def process_api_response(self, api_data: list, fetch_rewards: bool = True):
        # ... existing code unchanged ...

        print("Processing API response...")
        
        for event_data in api_data:
            title = event_data[0]
            yes_ids = event_data[1]  # List of YES token IDs
            no_ids = event_data[2]   # List of NO token IDs
            condition_ids = event_data[3]  # List of condition IDs
            rewards_daily_rates = event_data[4]
            rewards_min_sizes = event_data[5]
            rewards_max_spreads = event_data[6]
            min_ticks = event_data[7]

            # Skip events with no markets
            if not condition_ids:
                continue
                
            # Create event ID - using hash of title + first condition for uniqueness
            event_id = title
            #event_id = f"event_{hash(title + condition_ids[0]) & 0xffffffff}"
            
            markets = {}
            for i, (condition_id, yes_token_id, no_token_id, rewards_daily_rate, rewards_min_size, rewards_max_spread, min_tick) in enumerate(zip(condition_ids, yes_ids, no_ids, rewards_daily_rates, rewards_min_sizes, rewards_max_spreads, min_ticks)):
                market_id = condition_id  # Using condition_id as market_id
                
                # Create tokens
                yes_token = MarketToken(
                    condition_id=condition_id,
                    token_id=yes_token_id,
                    token_type="YES"
                )
                
                no_token = MarketToken(
                    condition_id=condition_id,
                    token_id=no_token_id,
                    token_type="NO"
                )
                
                # Fetch market rewards if requested
                rewards = MarketRewards(rewards_daily_rate, rewards_min_size, rewards_max_spread)
                
                # Create market
                market_title = f"{title} - Market {i+1}" if len(condition_ids) > 1 else title
                
                market = Market(
                    market_id=market_id,
                    title=market_title,
                    condition_id=condition_id,
                    yes_token=yes_token,
                    no_token=no_token,
                    rewards=rewards,
                    min_tick=min_tick
                )
                
                markets[market_id] = market
                
                # Store references
                self.markets[market_id] = market
                self.tokens[yes_token_id] = yes_token
                self.tokens[no_token_id] = no_token
            
            # Create event
            event = Event(
                event_id=event_id,
                title=title,
                markets=markets
            )
            
            self.events[event_id] = event
            
        print(f"Loaded {len(self.events)} events, {len(self.markets)} markets, {len(self.tokens)} tokens")

    def get_market_by_token(self, token_id: str) -> Optional[Market]:
        """Get market that contains a specific token"""
        token = self.tokens.get(token_id)
        if token:
            return self.markets.get(token.condition_id)
        return None
    
    def get_event_by_token(self, token_id: str) -> Optional[Event]:
        """Get event that contains a specific token"""
        market = self.get_market_by_token(token_id)
        if market:
            for event in self.events.values():
                if market.market_id in event.markets:
                    return event
        return None
    
    def get_token_info(self, token_id: str) -> Optional[Tuple[MarketToken, Market, Event]]:
        """Get complete information for a token"""
        token = self.tokens.get(token_id)
        if not token:
            return None
        
        market = self.get_market_by_token(token_id)
        if not market:
            return None
        
        event = self.get_event_by_token(token_id)
        if not event:
            return None
        
        return (token, market, event)
    def mark_event_interesting(self, event_id: str):
        self.active_events.add(event_id)


    def should_do_full_scan(self) -> bool:
        now = time.time()
        if now - self.last_full_scan >= self.FULL_SCAN_INTERVAL_SECONDS:
            self.last_full_scan = now
            return True
        return False

    def should_check_event_now(self, event_id: str, min_seconds_between: float = 30) -> bool:
        now = time.time()
        last_checked = self.recently_checked_events.get(event_id)  # ✅ safe — returns None if missing

        if last_checked is None or now - last_checked >= min_seconds_between:
            self.recently_checked_events[event_id] = now  # ✅ always record before returning
            return True
        return False
    
    def place_order(self, token_id: str, price: float, size: float, side: OrderSide, initial_market_bid: float, initial_market_ask: float, client) -> Optional[OrderRecord]:
        """Place an order for a specific token"""
        
        # Get complete token info
        token_info = self.get_token_info(token_id)
        if not token_info:
            print(f"Error: Token {token_id} not found in manager")
            return None
        
        token, market, event = token_info
        
        # Check if order meets minimum size requirement
        #if size < market.rewards.min_size:
            #print(f"Error: Order size {size} is below minimum {market.rewards.min_size} for market {market.title}")
            #return None
        
        # Cancel existing order for this token if any
        if token_id in self.active_orders:
            print(f"Cancelling existing order for token {token_id}")
            self.cancel_order(token_id, client)
        
        # Prepare order arguments
        order_args = OrderArgs(
            price=price,
            size=size,
            side=side.value,
            token_id=token_id,
            expiration=int(time.time()) + 240
        )
        
        try:
            # Create and post order
            signed_order = client.create_order(order_args)
            resp = client.post_order(signed_order, OrderType.GTD)
            
            if not resp.get('success', False):
                print(f"Order failed: {resp.get('errorMsg', 'Unknown error')}")
                return None
            
            # Create order record
            order_record = OrderRecord(
                order_id=resp['orderID'],
                asset_id=token_id,
                market_id=market.market_id,
                event_id=event.event_id,
                price=price,
                size=size,
                initial_market_bid=initial_market_bid,
                initial_market_ask=initial_market_ask,
                side=side,
                token_type=token.token_type,
                created_at=time.time(),
                expiration=order_args.expiration,
                status=OrderStatus.MATCHED if resp.get('status') == 'matched' else OrderStatus.PENDING,
                matched_amount=float(resp.get('takingAmount', 0)) if resp.get('takingAmount') else None,
                tx_hash=resp.get('transactionsHashes', [None])[0] if resp.get('transactionsHashes') else None,
                raw_response=resp
            )
            
            # Store order
            self.active_orders[token_id] = order_record
            self.order_id_mapping[resp['orderID']] = token_id
            # At the end of place_order, after self.active_orders[token_id] = order_record
            self.mark_event_interesting(event.event_id)
            print(f"Order placed: {order_record.order_id} for {token.token_type} token "
                  f"in market '{market.title}' at price ${price} for size {size}. Market bid: {initial_market_bid}, Market ask: {initial_market_ask}")
            self.subscribe_to_tokens_ids(event)
            return order_record
        
        except Exception as e:
            print(f"Error placing order for token {token_id}: {e}")
            return None    


    def cancel_order(self, token_id: str, client):
        """Cancel an active order"""
        if token_id in self.active_orders:
            order = self.active_orders[token_id]
            event_id = order.event_id 


            try:
                resp = client.cancel(order_id = order.order_id)  
                print(f"Cancelling order {order.order_id} for token {token_id}")
                print(f"Cancel response: {resp}")
                order.status = OrderStatus.CANCELLED
            except Exception as e:
                print(f"Error cancelling order {order.order_id}: {e}")
            finally:
                # Cleanup
                self.order_id_mapping.pop(order.order_id, None)
                del self.active_orders[token_id]
                print(f"Order {order.order_id} removed from tracking")
                self.check_and_deactivate_event(event_id)   # ✅

    def cancel_orders(self, token_ids: list, client):
        """Cancel multiple active orders using bulk cancellation"""
        
        # Collect order_ids for the given token_ids
        order_ids_to_cancel = []
        token_to_order_map = {}  # Map order_id -> token_id
        event_ids_affected = set() 

        for token_id in token_ids:
            if token_id in self.active_orders:
                order = self.active_orders[token_id]
                order_ids_to_cancel.append(order.order_id)
                token_to_order_map[order.order_id] = token_id
                event_ids_affected.add(order.event_id)
            else:
                print(f"Token {token_id} not found in active orders")
        
        if not order_ids_to_cancel:
            print("No active orders found to cancel")
            return
        
        # Perform bulk cancellation
        try:
            print(f"Cancelling {len(order_ids_to_cancel)} orders: {order_ids_to_cancel}")
            resp = client.cancel_orders(order_ids_to_cancel)
            print(f"Bulk cancel response: {resp}")
            
            # Update each order's status and cleanup
            for order_id in order_ids_to_cancel:
                token_id = token_to_order_map[order_id]
                order = self.active_orders[token_id]
                order.status = OrderStatus.CANCELLED
                
                # Cleanup
                self.order_id_mapping.pop(order_id, None)
                del self.active_orders[token_id]
                print(f"Order {order_id} (token {token_id}) cancelled and removed from tracking")
                
            # ✅ Check all affected events after bulk removal
            for event_id in event_ids_affected:
                self.check_and_deactivate_event(event_id)
            
        except Exception as e:
            print(f"Error in bulk cancellation: {e}")


    def get_price(self, token_id: str):
        book_param = BookParams(token_id=token_id)
        Book = self.client.get_order_books(params= [book_param])
        data = float(Book[0].asks[-1].price) if Book and Book[0].asks else 0.999
        return data

    def hedging(self, event: Event, client):
        orders_filled = []
        filled_asset_ids = []
        total_cost = 0.0
        all_asset_ids = event.get_all_no_tokens()

        all_order_ids = [order.order_id for order in self.active_orders.values() 
                        if order.event_id == event.event_id]
        if not all_order_ids:
            return True

        # Use WS-updated matched_amount preferentially
        for token_id in all_asset_ids:
            if token_id in self.active_orders:
                order = self.active_orders[token_id]
                matched = order.matched_amount or 0.0

                # Light fallback poll only if WS data is stale (>60s)
                if matched == 0 and (order.last_checked is None or time.time() - order.last_checked > 60):
                    try:
                        resp = client.get_order(order.order_id)
                        matched = float(resp.get('size_matched', 0))
                        order.matched_amount = matched
                        order.last_checked = time.time()
                    except Exception as e:
                        print(f"[Hedge] Poll failed for {token_id}: {e}")
                        continue

                # ✅ Subtract already hedged shares to avoid double hedging
                already_hedged = self.get_position(token_id)
                net_unhedged = max(0.0, matched - already_hedged)

                if net_unhedged > 0:
                    orders_filled.append([token_id, net_unhedged])
                    total_cost += net_unhedged * order.price
                    filled_asset_ids.append(token_id)

        if not orders_filled:
            self.cleanup_expired_orders()
            return True  # No fills → WS will handle price protection

        # Cancel all maker orders in the event to prevent over-hedge
        self.cancel_orders(all_asset_ids, client)

        size_to_be_hedged = max([m[1] for m in orders_filled]) if orders_filled else 0

        # Hedge filled orders (partial fills)
        for token_id, filled in orders_filled:
            if abs(size_to_be_hedged - filled) > 1e-7:
                price = self.get_price(token_id)
                remaining = size_to_be_hedged - filled
                amount_usd = remaining * price
                if amount_usd < 1 or remaining < 5:
                    print(f"Cannot hedge remaining {remaining} on {token_id} (USD {amount_usd:.2f})")
                    total_cost += amount_usd
                    continue
                order_args = OrderArgs(
                    price=price,
                    size=remaining,
                    side="BUY",
                    token_id=token_id,
                    expiration=int(time.time()) + 700
                )
                signed = client.create_order(order_args)
                resp = client.post_order(signed, OrderType.GTD)
                making = float(resp.get('makingAmount', remaining * price))
                total_cost += making
                self.update_position(token_id, remaining, making)  # ✅

        # Hedge unfilled orders
        not_filled_ids = [tid for tid in all_asset_ids if tid not in filled_asset_ids]
        if not_filled_ids:
            book_params = [BookParams(token_id=tid) for tid in not_filled_ids]
            books = client.get_order_books(params=book_params)

            post_orders_args_list = []
            orders_info = []

            for book in books:
                token_id = book.asset_id
                if not book.asks:
                    continue
                ask = book.asks[-1]
                price = float(ask.price)
                usd_cost = size_to_be_hedged * price
                if usd_cost < 1:
                    print(f"Cannot hedge {token_id} - USD {usd_cost:.2f} too small")
                    total_cost += usd_cost
                    continue

                order_args = OrderArgs(
                    price=price,
                    size=size_to_be_hedged,
                    side="BUY",
                    token_id=token_id,
                    expiration=int(time.time()) + 70
                )
                signed = client.create_order(order_args)
                post_orders_args_list.append(
                    PostOrdersArgs(order=signed, orderType=OrderType.GTD, postOnly=False)
                )
                orders_info.append({'token_id': token_id, 'price': price, 'size': size_to_be_hedged})

            # Batch posting & cost calc
            BATCH_SIZE = 15
            total_successful_orders = 0

            for i in range(0, len(post_orders_args_list), BATCH_SIZE):
                batch = post_orders_args_list[i:i+BATCH_SIZE]
                batch_info = orders_info[i:i+BATCH_SIZE]
                resp_list = client.post_orders(batch)

                for j, resp in enumerate(resp_list):
                    info = batch_info[j]
                    if resp.get('success'):
                        total_successful_orders += 1
                        making = float(resp.get('makingAmount', info['size'] * info['price']))
                        total_cost += making
                        self.update_position(info['token_id'], info['size'], making)  # ✅

                        # Handle partial fill
                        taking = float(resp.get('takingAmount', 0))
                        if taking < info['size']:
                            remaining = info['size'] - taking
                            rem_usd = remaining * info['price']
                            if rem_usd >= 1 and remaining >= 5:
                                # Re-hedge
                                order_args = OrderArgs(
                                    price=info['price'],
                                    size=remaining,
                                    side="BUY",
                                    token_id=info['token_id'],
                                    expiration=int(time.time()) + 70
                                )
                                signed2 = client.create_order(order_args)
                                resp2 = client.post_order(signed2, OrderType.GTD)
                                making2 = float(resp2.get('makingAmount', remaining * info['price']))
                                total_cost += making2
                                self.update_position(info['token_id'], remaining, making2)  # ✅
                            else:
                                total_cost += rem_usd
                    else:
                        print(f"Hedge failed for {info['token_id']}: {resp.get('errorMsg')}")

            # Telegram notification
            if total_successful_orders > 0:
                pnl = self.get_event_pnl(event)
                message = (
                    f"Trade Notification:\n"
                    f"Event: {event.title}\n"
                    f"Market Buy Hedge executed for {total_successful_orders} tokens\n"
                    f"Cost of Hedge: {total_cost:.2f}\n"
                    f"Total Size per token: {size_to_be_hedged}\n"
                    f"Theoretical Profit: {pnl['theoretical_profit']:.2f}"  # ✅ uses ledger
                )
                Telegram_url = f"https://api.telegram.org/bot{tg_token}/sendMessage?chat_id={chat_id}&text={message}"
                requests.get(Telegram_url)

        return True
    
    def get_orders_by_event(self, event_id: str) -> List[OrderRecord]:
        """Get all orders for a specific event"""
        return [
            order for order in self.active_orders.values() 
            if order.event_id == event_id
        ]
    
    def get_orders_by_market(self, market_id: str) -> List[OrderRecord]:
        """Get all orders for a specific market"""
        return [
            order for order in self.active_orders.values() 
            if order.market_id == market_id
        ]
    
    def filter_events(self, filter_func) -> List[Event]:
        """Filter events based on custom criteria"""
        filtered = []
        for event in self.events.values():
            if filter_func(event):
                filtered.append(event)
        return filtered
    
    def get_all_no_tokens(self) -> List[MarketToken]:
        """Get all NO tokens"""
        return [
            token for token in self.tokens.values() 
            if token.token_type == "NO"
        ]
    
    def get_all_yes_tokens(self) -> List[MarketToken]:
        """Get all YES tokens"""
        return [
            token for token in self.tokens.values() 
            if token.token_type == "YES"
        ]
    
    def get_opposite_token(self, token_id: str) -> Optional[MarketToken]:
        """Given a token ID, get the opposite token (YES ↔ NO) in the same market"""
        market = self.get_market_by_token(token_id)
        if market:
            return market.get_other_token(token_id)
        return None
    
    def get_market_summary(self, market_id: str) -> Dict:
        """Get summary of a market including any active orders"""
        market = self.markets.get(market_id)
        if not market:
            return {}
        
        yes_orders = self.get_orders_by_token(market.yes_token.token_id)
        no_orders = self.get_orders_by_token(market.no_token.token_id)
        
        return {
            "market": market.to_dict(),
            "yes_orders": [order.to_dict() for order in yes_orders],
            "no_orders": [order.to_dict() for order in no_orders]
        }
    
    def get_orders_by_token(self, token_id: str) -> List[OrderRecord]:
        """Get all orders for a specific token (should be max 1)"""
        order = self.active_orders.get(token_id)
        return [order] if order else []
    
    def cleanup_expired_orders(self):
        """Remove expired orders from tracking"""
        expired_tokens = []
        for token_id, order in self.active_orders.items():
            if order.is_expired():
                expired_tokens.append(token_id)
        event_ids_affected = set()

        for token_id in expired_tokens:
            order = self.active_orders[token_id]
            event_ids_affected.add(order.event_id)   # ✅
            self.order_id_mapping.pop(order.order_id, None)
            del self.active_orders[token_id]
            print(f"Cleaned up expired order {order.order_id}")

        # ✅ Check all affected events after expiry cleanup
        for event_id in event_ids_affected:
            self.check_and_deactivate_event(event_id)
    
    def cleanup_expired_orders_of_event(self, event: Event):
        """Remove expired orders from tracking"""
        expired_tokens = []
        for token_id, order in self.active_orders.items():
            if order.is_expired() and order.event_id == event.event_id:
                expired_tokens.append(token_id)
        
        event_ids_affected = set()
        
        for token_id in expired_tokens:
            order = self.active_orders[token_id]
            event_ids_affected.add(order.event_id)   # ✅
            self.order_id_mapping.pop(order.order_id, None)
            del self.active_orders[token_id]
            print(f"Cleaned up expired order {order.order_id}")

        # ✅ Check all affected events after expiry cleanup
        for event_id in event_ids_affected:
            self.check_and_deactivate_event(event_id)

    def update_position(self, token_id: str, shares: float, cost: float):
        """Record a position after a successful hedge buy"""
        if token_id not in self.positions:
            self.positions[token_id] = {
                "shares": 0.0,
                "avg_cost": 0.0,
                "total_cost": 0.0
            }
        
        pos = self.positions[token_id]
        new_total_shares = pos["shares"] + shares
        new_total_cost = pos["total_cost"] + cost

        pos["shares"] = new_total_shares
        pos["total_cost"] = new_total_cost
        pos["avg_cost"] = new_total_cost / new_total_shares if new_total_shares > 0 else 0.0

        print(f"[Position] {token_id}: {new_total_shares:.2f} shares @ avg {pos['avg_cost']:.4f} | total cost {new_total_cost:.2f}")

    def get_position(self, token_id: str) -> float:
        """Get current share count for a token, 0 if none"""
        return self.positions.get(token_id, {}).get("shares", 0.0)

    def clear_position(self, token_id: str):
        """Clear position after market resolves"""
        if token_id in self.positions:
            del self.positions[token_id]
            print(f"[Position] Cleared position for {token_id}")

    def get_event_pnl(self, event: Event) -> Dict:
        """Summarise P&L across all legs of an event"""
        total_cost = 0.0
        total_shares = 0.0
        legs = {}

        for market in event.markets.values():
            token_id = market.no_token.token_id
            pos = self.positions.get(token_id, {})
            shares = pos.get("shares", 0.0)
            cost = pos.get("total_cost", 0.0)
            total_cost += cost
            total_shares = max(total_shares, shares)  # max since only one leg wins
            legs[token_id] = {"shares": shares, "cost": cost}

        # In neg-risk: profit = shares * (n_legs - 1) - total_cost
        n_legs = len(event.markets)
        theoretical_profit = total_shares * (n_legs - 1) - total_cost

        return {
            "event": event.title,
            "total_cost": round(total_cost, 4),
            "max_shares": round(total_shares, 4),
            "theoretical_profit": round(theoretical_profit, 4),
            "legs": legs
        }        

In [4]:
def try_place_new_makers(manager: TradingManager, event: Event, client):
    no_token_ids = event.get_all_no_tokens()
    if not no_token_ids:
        return

    book_params = [BookParams(token_id=tid) for tid in no_token_ids]
    try:
        books = client.get_order_books(params=book_params)
    except Exception as e:
        print(f"Order books fetch failed for {event.title}: {e}")
        return

    if len(books) != len(no_token_ids):
        return

    valid_books = []
    price_sum = 0.0
    min_ask_size = float('inf')
    min_ask_price = 1.0

    for book in books:
        if not book.asks:
            return   # skip whole event if any missing asks
        ask = book.asks[-1]
        p = float(ask.price)
        s = float(ask.size)
        # best bid from book if available
        best_bid = float(book.bids[-1].price) if book.bids else 0.0

        price_sum += p
        min_ask_size = min(min_ask_size, s)
        min_ask_price = min(min_ask_price, p)

        # Create a MarketToken for this NO token and store/update it in manager.tokens
        if book.asset_id not in manager.tokens:
            market = manager.get_market_by_token(book.asset_id)
            condition_id = market.market_id if market else ""
            token = MarketToken(
                condition_id=condition_id,
                token_id=book.asset_id,
                token_type="NO",
                best_bid=best_bid,
                best_ask=p
            )
            manager.tokens[book.asset_id] = token
        else:
            # update snapshot values
            token = manager.tokens[book.asset_id]
            token.best_ask = p
            token.best_bid = best_bid

        valid_books.append(book)

    if min_ask_size < 5 or min_ask_price * min_ask_size < 1:
        return

    # Very simple size — can be improved later (balance-based, etc.)
    target_size = 20.0   # ← your old fixed value; consider making dynamic

    collateral_resp = client.get_balance_allowance(
        params=BalanceAllowanceParams(asset_type=AssetType.COLLATERAL)
    )
    cash_usdc = float(collateral_resp["balance"]) / 4000000
    target_size = min(target_size, cash_usdc / len(no_token_ids))

    if target_size < 5:
        return

    for book in valid_books:
        market = manager.get_market_by_token(book.asset_id)
        if not market:
            continue

        min_tick = market.min_tick
        initial_ask = float(book.asks[-1].price)
        initial_bid = float(book.bids[-1].price) if book.bids else 0.0

        theoretical = len(no_token_ids) - 1 - (price_sum - initial_ask) - min_tick - 0.001
        our_bid = min(theoretical, initial_bid + min_tick)
        our_bid = math.floor(our_bid / min_tick) * min_tick

        # Only place if we would be at or better than current best bid
        if our_bid >= initial_bid - 1e-6:   # small epsilon for float
            placed = manager.place_order(
                token_id=book.asset_id,
                price=our_bid,
                size=target_size,
                side=OrderSide.BUY,
                initial_market_bid=initial_bid,
                initial_market_ask=initial_ask,
                client=client
            )
            if placed:
                manager.mark_event_interesting(event.event_id)
                print(f"Placed new maker order in {event.title} @ {our_bid:.4f}")



In [ ]:
def market_making():
    # Assume client is your Polymarket client instance
    # Assume tg_token, chat_id defined
    # global tg_token, chat_id
    # tg_token = "YOUR_TG_TOKEN"
    # chat_id = "YOUR_CHAT_ID"
    manager = TradingManager(
        client=client, 
        api_key=api_creds.api_key, 
        api_secret=api_creds.api_secret, 
        api_passphrase=api_creds.api_passphrase
    )
    # ✅ Register signal handler from main thread
    manager.register_signal_handler()

    raw_data = get_YES_NO_And_Condition(fetch_count=500)
    manager.process_api_response(raw_data)

    iteration = 0

    while manager.ws_running:
        iteration += 1
        now_str = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
        print(f"Iteration {iteration} ─ {now_str} ─ Active events: {len(manager.active_events)}")

        # Check WS heartbeat
        if time.time() - manager.last_ws_message_time > 120:
            print("[WS] No messages for 2 min → reconnecting")
            manager.stop_websocket()
            time.sleep(1)
            manager.start_websocket()

        # ✅ Periodic market data refresh
        if manager.should_refresh_markets():
            manager.refresh_markets(fetch_count=500)

        # A. Periodic scan for NEW opportunities (excluding active)
        if manager.should_do_full_scan():
            print("  → Periodic scan for NEW maker opportunities "
                  f"(excluding {len(manager.active_events)} already active events)")
            active_event_ids = manager.active_events.copy()
            candidates = [
                event for event_id, event in manager.events.items()
                if event_id not in active_event_ids
            ]
            for event in candidates:
                try_place_new_makers(manager, event, client)

        if not manager.active_events:
            print("  No active positions → longer sleep")
            time.sleep(10)
            continue

        print(f"  Checking {len(manager.active_events)} events with live positions...")

        for event_id in list(manager.active_events):
            event = manager.events.get(event_id)
            if not event:
                manager.active_events.discard(event_id)
                continue
            

            # ✅ Rate limit per-event checks to avoid hammering the API
            if not manager.should_check_event_now(event_id, min_seconds_between=30):
                print(f"  Skipping {event_id} — checked too recently")
                continue
            
            # Always hedge and place makers
            manager.hedging(event, client)
            try_place_new_makers(manager, event, client)

        # Cleanup
        manager.cleanup_expired_orders()

        sleep_sec = 5 if manager.active_events else 20
        print(f"  Sleeping {sleep_sec}s ...\n")
        time.sleep(sleep_sec)

# Run the bot


In [ ]:
if __name__ == "__main__":
    # Define client, tg_token, chat_id here
    # client = YourPolymarketClient()
    # tg_token = ""
    # chat_id = ""
    market_making()

[WS] Thread started
[Market WS] Thread started
[signal_handler] Registered — press Ctrl+C to stop gracefully
[Market WS] Connected → re-subscribing to active tokens
[Market WS] No active tokens to subscribe to on connect
[WS] Connected → subscribing to user channel
[WS] Ping thread started
[WS] RAW: PONG
[WS] JSON decode failed: Expecting value: line 1 column 1 (char 0)
Processing API response...
Loaded 170 events, 1285 markets, 2570 tokens
Iteration 1 ─ 2026-03-12 12:34:38 ─ Active events: 0
  → Periodic scan for NEW maker opportunities (excluding 0 already active events)
[WS] RAW: {"id":"0xd3f90c8df69f0c05b0e31d93a0fb57280352d33fa827775de90ba2ea90c13cf4", "owner":"6fcf5b7c-4cd4-1eea-4133-4d9c20fe0c94", "market":"0x645207afaa613a667bf1a7f1d4feff15fbe74a54642e897a8b07b6d9ee3ac07a", "asset_id":"37759118010832272182075273998299383689504209579695244279398547048050420123363", "side":"BUY", "order_owner":"6fcf5b7c-4cd4-1eea-4133-4d9c20fe0c94", "original_size":"20", "size_matched":"0", "pric